# Objective -    For exploratoy analysis of the equity chart for selected combinations

######  Description -  The code will calculate correlations and basic trading metrics like Sharpe ratio, VAR for each combination


#####   Date - 15th Dec 2021

######  Author - Avinash

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
path = "<local-path>"

In [ ]:
df_48_post_model= pd.read_excel(path + "Model vs Base - with Nov Validation - 48 Combs 20211214 V1.xlsx", sheet_name='Trade Level Data-Filtered(model')


df_48_base= pd.read_excel(path + "Model vs Base - with Nov Validation - 48 Combs 20211214 V1.xlsx", sheet_name='Trade Level Data - (Base)',skiprows = 4)

df_18_base= pd.read_excel(path + "The firm Final Selected Combinations(18 Combs) - Addition of Oct-Nov - Model vs Base 20211214 V1.xlsx", sheet_name='Data - with DD threshold(-2000)')


# 1..Getting day, month, year and rolling up the data at day level

In [ ]:
df_18_base['date']=pd.to_datetime(df_18_base['Opening Date'],format = '%Y-%m-%d %H:%M:%S')
df_18_base['Year'] = df_18_base['date'].dt.year
df_18_base['Month'] = df_18_base['date'].dt.month
df_18_base['Day'] = df_18_base['date'].dt.day



df_18_base_day = pd.DataFrame(df_18_base.groupby(by=['Strategy','Currency','Condition','Timeframe','Year','Month','Day'])['Profit','PL CuttOff + Dynamic Sizing','P/L using Dynamic Sizing'].sum())
df_18_base_day = df_18_base_day.reset_index()
df_18_base_day



In [ ]:
df_48_base['date']=pd.to_datetime(df_48_base['Opening Date'],format = '%Y-%m-%d %H:%M:%S')
df_48_base['Year'] = df_48_base['date'].dt.year
df_48_base['Month'] = df_48_base['date'].dt.month
df_48_base['Day'] = df_48_base['date'].dt.day



df_48_base_day = pd.DataFrame(df_48_base.groupby(by=['Strategy','Currency','Condition','Timeframe','Year','Month','Day'])['Profit'].sum())
df_48_base_day = df_48_base_day.reset_index()
df_48_base_day



In [ ]:
df_48_post_model['date']=pd.to_datetime(df_48_post_model['Opening Date'],format = '%Y-%m-%d %H:%M:%S')
df_48_post_model['Year'] = df_48_post_model['date'].dt.year
df_48_post_model['Month'] = df_48_post_model['date'].dt.month
df_48_post_model['Day'] = df_48_post_model['date'].dt.day



df_48_model_day = pd.DataFrame(df_48_post_model.groupby(by=['Strategy','Currency','Condition','Timeframe','Year','Month','Day'])['Profit'].sum())
df_48_model_day = df_48_model_day.reset_index()
df_48_model_day



In [ ]:
df_48_base_day['Unique_FlAG'] = df_48_base_day['Strategy'] + "_" + df_48_base_day['Currency'] + "_"+ df_48_base_day['Condition'] 
iteration_list = df_48_base_day['Unique_FlAG'].unique()

#iteration_list = iteration_list[1:5]
#iteration_list

In [ ]:
df_48_model_day['Unique_FlAG'] = df_48_model_day['Strategy'] + "_" + df_48_model_day['Currency'] + "_"+ df_48_model_day['Condition'] 
iteration_list = df_48_model_day['Unique_FlAG'].unique()

#iteration_list = iteration_list[1:5]
#iteration_list

In [ ]:
df_18_base_day['Unique_FlAG'] = df_18_base_day['Strategy'] + "_" + df_18_base_day['Currency'] + "_"+ df_18_base_day['Condition'] 
iteration_list = df_18_base_day['Unique_FlAG'].unique()

#iteration_list = iteration_list[1:5]
iteration_list

In [ ]:
df_48_model_day

# Loop for getting the correlations by combinations .. correlation will be calculate for common days for each combinations

In [ ]:
i=0

results_18_dynamic= pd.DataFrame()

for outer_loop in iteration_list:
    i+=1
    print(outer_loop)
    
    print(i)
    
    for inner_loop in iteration_list:
        
        #print(inner_loop
        temp = df_18_base_day[df_18_base_day['Unique_FlAG']==outer_loop]
        temp2 = df_18_base_day[df_18_base_day['Unique_FlAG']==inner_loop]
        
        
        temp = temp.reset_index()
        temp2 = temp2.reset_index()
        
        
        
        final_temp = pd.merge(left=temp,
                            right=temp2,
                            left_on=['Day','Month','Year'],
                            right_on=['Day','Month','Year'],
                            how = 'inner',suffixes=('_left', '_right'))


        res = final_temp[['P/L using Dynamic Sizing_right','P/L using Dynamic Sizing_left']].corr()
        res.iloc[0,1]
        
        test = pd.DataFrame()
        
        test.loc[1,'Condition_right'] = temp.loc[0,'Condition']
        test.loc[1,'Strategy_right'] = temp.loc[0,'Strategy']
        test.loc[1,'Currency_right'] = temp.loc[0,'Currency']
        
        
        test.loc[1,'Condition_left'] = temp2.loc[0,'Condition']
        test.loc[1,'Strategy_left'] = temp2.loc[0,'Strategy']
        test.loc[1,'Currency_left'] = temp2.loc[0,'Currency']
        
        
        
        test.loc[1,'Combination_left'] = inner_loop
        
        test.loc[1,'Combination_right'] = outer_loop
        test.loc[1,'Common_days'] = len(final_temp)
        test.loc[1,'Correlation'] = res.iloc[0,1]
        
        results_18_dynamic = results_18_dynamic.append(test)
        


In [ ]:
results_18_base

In [ ]:
results

In [ ]:
results.to_excel("Correlation_results_48_combinations_base.xlsx")

In [ ]:
results

In [ ]:
results_base

In [ ]:
##### Writing the correlation results in an excel file

with pd.ExcelWriter('Correlation_analysis_20211215_v1.xlsx') as writer:
    results.to_excel(writer, sheet_name= "48_Comb_model")
    results_base.to_excel(writer, sheet_name= "48_Comb_base")
    results_18_dynamic.to_excel(writer, sheet_name= "18_Comb_dynamic")
    results_18_base.to_excel(writer, sheet_name= "18_Comb_base")   
    

# Using Quantstats to get trading statistics

In [ ]:
import quantstats as qs

import pandas as pd
import numpy as np
import os
import datetime as dt
import ta
import talib
from IPython.display import clear_output
import matplotlib.pyplot as pl

qs.extend_pandas()

### Creating the date variable from year month and day

In [ ]:
df_48_model_day['dateInt']=df_48_model_day['Year'].astype(str) + df_48_model_day['Month'].astype(str).str.zfill(2)+ df_48_model_day['Day'].astype(str).str.zfill(2)

df_48_model_day['Date'] = pd.to_datetime(df_48_model_day['dateInt'], format='%Y%m%d')


In [ ]:
df_48_base_day['dateInt']=df_48_base_day['Year'].astype(str) + df_48_base_day['Month'].astype(str).str.zfill(2)+ df_48_base_day['Day'].astype(str).str.zfill(2)

df_48_base_day['Date'] = pd.to_datetime(df_48_base_day['dateInt'], format='%Y%m%d')


In [ ]:
df_18_base_day['dateInt']=df_18_base_day['Year'].astype(str) + df_18_base_day['Month'].astype(str).str.zfill(2)+ df_18_base_day['Day'].astype(str).str.zfill(2)

df_18_base_day['Date'] = pd.to_datetime(df_18_base_day['dateInt'], format='%Y%m%d')


In [ ]:
df_18_base_day

### loopp for getting the metrics by combinations

In [ ]:
i=0

results= pd.DataFrame()


df=df_18_base_day

iteration_list = df['Unique_FlAG'].unique()

path = "<local-path>"

for outer_loop in iteration_list:
    
        try:
        
            i+=1
            print(i)
            temp = df[df['Unique_FlAG']==outer_loop]
            temp=temp.reset_index()



            Equity = temp['Profit']/20000    ### converting to 0.5% risk for the final calcualtion

            test = pd.DataFrame()

            test.loc[1,'Combination'] = outer_loop
            test.loc[1,'Condition'] = temp.loc[1,'Condition']
            test.loc[1,'Strategy'] = temp.loc[1,'Strategy']
            test.loc[1,'Currency'] = temp.loc[1,'Currency']

            test.loc[1,'sharp_ratio'] = qs.stats.sharpe(Equity)
            test.loc[1,'VAR'] = qs.stats.value_at_risk(Equity)
            test.loc[1,'value_at_risk'] = qs.stats.value_at_risk(Equity)
            test.loc[1,'avg_return'] = qs.stats.avg_return(Equity)
            test.loc[1,'max_drawdown'] = qs.stats.max_drawdown(Equity)

            #test.loc[1,'CAGR'] = qs.stats.cagr(Equity)

            var = outer_loop.replace('/','_')

            temp = temp.set_index('Date')
            qs.reports.html(temp['P/L using Dynamic Sizing']/20000,title = outer_loop,output=path + var + "_dynamic.html")


            results = results.append(test)
            
            clear_output()
        except :
            print("error at " + outer_loop)
            

results.to_excel(path + "Dynamic_Metrics_by_combination.xlsx")
        

In [ ]:
qs.stats.max_drawdown(Equity)

In [ ]:
var